
# Part 1 — The Hull Truth Forum Scraper

Scrapes Gulf-Council / fisheries threads from *thehulltruth.com* (the Gulf Coast
subforum and its nested children), keeps only **policy-relevant** threads
published in the **last 10 years**, and writes readable text plus structured
metadata into the shared `Data_Repository`.

## Setup — Install & Imports

Installs the third-party packages and imports the standard-library + scraping
modules (BeautifulSoup for HTML, curl_cffi for Cloudflare-aware requests) used
throughout Part 1.

In [ ]:
!pip install pymupdf beautifulsoup4 duckduckgo-search curl_cffi

import argparse
import csv
import json
import random
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.parse import urljoin, urlparse, urldefrag

from bs4 import BeautifulSoup
from curl_cffi import requests as cr

## Configuration

The target site, all output paths (everything lands under `Data_Repository`),
crawl limits, the 10-year data window, and the keyword sets that decide which
threads count as *relevant* and which clear the stricter *policy* gate.

In [ ]:
BASE_URL = "https://www.thehulltruth.com/"
DOMAIN   = "www.thehulltruth.com"

# Ordered broadly newest-to-oldest. fetch() probes what the installed curl_cffi
# actually supports and drops the rest into _BAD_IMPERSONATES at runtime.
IMPERSONATE_OPTIONS = [
    "chrome136", "chrome133a", "chrome131", "chrome124", "chrome123",
    "chrome120", "chrome119", "chrome116", "chrome110", "chrome107",
    "chrome104", "chrome101", "chrome100", "chrome99",
    "edge101", "edge99",
    "safari18_0", "safari17_0", "safari15_5", "safari15_3",
]

BROWSER_HEADERS = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9",
    "Cache-Control": "max-age=0",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Upgrade-Insecure-Requests": "1",
}

ROOT         = Path.cwd().resolve()
DATA         = ROOT / "Data_Repository"
RAW_TXT_DIR  = DATA / "scraped_text" / "blogs_forums"
METADATA_DIR = DATA / "metadata" / "blogs_forums"
RECORDS_DIR  = DATA / "records"
STATE_FILE   = DATA / "source_registry.json"


# kept (we can't prove they're old).
DATA_WINDOW_YEARS = 10

# Metadata schema constants
SOURCE_TYPE = "forum"
SOURCE_NAME = "The Hull Truth"
METADATA_FIELDS = (
    "Document_ID", "Source_Type", "Source_Name", "URL", "Title",
    "Published_Date", "Location",
)

WORKERS   = 8
TIMEOUT   = 45
RETRIES   = 4
DELAY_MIN = 0.4
DELAY_MAX = 1.2


# "gulf-coast" matches "gulf-coast-31" and any nested "gulf-coast-*" child.
SUBFORUM_ALLOWLIST: list[str] = ["gulf-coast"]

# Hard caps
TARGET_THREADS       = 7000  # download goal
MAX_LIST_PAGES       = 5000
MAX_THREAD_PAGES     = 500

# Gulf Council / fisheries relevance filter.
# URL slugs on this site are lowercase-hyphenated.
RELEVANCE_KEYWORDS: tuple[str, ...] = (
    # Management bodies + law
    "gulf-council", "gmfmc", "nmfs", "noaa", "magnuson", "msa",
    "fwc", "tpwd", "lwf", "ldwf", "wlf",
    "council", "amendment", "framework", "regulation", "regulations",
    "rule", "rulemaking", "ifq", "quota", "allocation",
    "season", "closure", "closed", "reopen", "open-season",
    "bag-limit", "bag", "size-limit", "slot-limit", "slot",
    "stock-assessment", "overfishing", "overfished", "rebuilding",
    "fmp", "mrip", "mrfss", "sefsc", "for-hire", "charter",
    "commercial", "recreational", "rec", "fisheries", "fishery",
    "management", "enforcement", "poach", "poaching", "citation",
    "by-catch", "bycatch", "discard", "release-mortality",
    "descending-device", "descender", "circle-hook", "venting",
    "barotrauma", "red-tide",
    # Gulf-relevant species
    "snapper", "red-snapper", "vermilion", "mutton", "yellowtail",
    "lane-snapper", "mangrove-snapper", "grey-snapper",
    "grouper", "gag", "red-grouper", "black-grouper", "scamp",
    "amberjack", "aj", "triggerfish", "hogfish", "porgy",
    "kingfish", "king-mackerel", "mackerel", "spanish-mackerel",
    "cobia", "ling", "tarpon", "redfish", "red-drum",
    "trout", "speckled-trout", "specks", "flounder", "pompano",
    "sheepshead", "black-drum", "jack-crevalle", "dorado", "dolphin", "tuna", "yellowfin",
    "blackfin", "wahoo", "marlin", "sailfish", "swordfish",
    "shark", "snook", "tripletail", "bonito", "blackfin-tuna",
    "reef-fish", "pelagic", "billfish", "shrimp", "stone-crab",
    "spiny-lobster", "lobster", "scallop", "oyster",
    # Gulf regions / waters
    "gulf-of-mexico", "gulf-coast", "florida-keys", "keys",
    "tampa", "destin", "panama-city", "pensacola", "orange-beach",
    "dauphin", "biloxi", "venice", "grand-isle", "cocodrie",
    "galveston", "port-aransas", "south-padre", "naples",
    "panhandle", "big-bend", "florida-middle-grounds", "middle-grounds",
    "eez", "federal-waters", "state-waters", "nine-mile", "9-mile",
    # Fishing activity (permissive, but confined to gulf-coast subforum)
    "fishing", "angler", "catch", "caught", "limit", "tournament",
    "offshore", "nearshore", "inshore", "reef", "wreck", "bottom",
    "trolling", "chunking", "jigging", "live-bait", "report",
    "trip-report", "fishing-report",
)

EXCLUDE_KEYWORDS: tuple[str, ...] = (
    "for-sale", "price-reduced", "prop-for-sale", "engine-for-sale",
    "classified", "classifieds",
    # Short tokens — matched only as whole tokens, never as substrings
    "fs", "wtb", "wts", "sold",
)


POLICY_KEYWORDS: tuple[str, ...] = (
    # Management bodies / law (precise agency names — no bare "council")
    "gulf-council", "gulf council", "gmfmc", "nmfs", "noaa", "fwc",
    "tpwd", "ldwf", "lwf", "wlf", "sefsc", "magnuson", "msa",
    # Regulatory instruments — only high-precision terms
    "amendment", "regulation", "regulations", "rulemaking",
    "ifq", "quota", "allocation", "fmp", "mrip", "mrfss",
    # Seasons / closures (multi-word forms only)
    "closure", "reopen", "open-season", "closed-season",
    "season-closure", "season-opening",
    # Catch limits (multi-word forms only)
    "bag-limit", "size-limit", "slot-limit", "trip-limit", "vessel-limit",
    # Stock assessment / status (dropped "rebuilding" — engine rebuilding)
    "stock-assessment", "overfishing", "overfished",
    # Sector classifications (policy context)
    "for-hire",
    # Bycatch / conservation gear
    "by-catch", "bycatch", "release-mortality",
    "descending-device", "descender", "venting-tool", "barotrauma",
    # Enforcement
    "enforcement", "poaching",
    # Public input
    "public-comment", "comment-period", "public-hearing", "public-meeting",
    "testimony",
    # FEI / Ecosystem-based fishery management
    # (dropped bare "sustainability"/"sustainable" — used too casually)
    "ecosystem-based-management", "ebm", "ecosystem-assessment",
    "fishery-management", "fisheries-management",
    "sustainable-fishery", "sustainable-fisheries",
    # Gulf-specific issues with policy weight
    "red-tide",
)

# Precompile keyword sets for fast matching.
_INCLUDE_SET = frozenset(k.lower() for k in RELEVANCE_KEYWORDS)
_EXCLUDE_SET = frozenset(k.lower() for k in EXCLUDE_KEYWORDS)
_POLICY_SET  = frozenset(k.lower() for k in POLICY_KEYWORDS)

## HTTP Fetcher

A Cloudflare-aware downloader: it rotates Chrome / Edge / Safari TLS
fingerprints, retries with exponential backoff, and throttles politely so the
site does not block the crawl.

In [ ]:
_thread_local = threading.local()
_BAD_IMPERSONATES: set[str] = set()
_BAD_LOCK = threading.Lock()

def _available_impersonates() -> list[str]:
    with _BAD_LOCK:
        return [o for o in IMPERSONATE_OPTIONS if o not in _BAD_IMPERSONATES]

def _is_unsupported_err(exc: BaseException) -> bool:
    name = type(exc).__name__.lower()
    msg = str(exc).lower()
    return "impersonate" in name or "not supported" in msg or "unknown impersonate" in msg

def _session_for(imp: str) -> cr.Session:
    key = f"sess_{imp}"
    s = getattr(_thread_local, key, None)
    if s is None:
        s = cr.Session(impersonate=imp, timeout=TIMEOUT)
        setattr(_thread_local, key, s)
    return s

def _polite():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def fetch(url: str) -> Optional[str]:
    url, _ = urldefrag(url)
    headers = dict(BROWSER_HEADERS)
    if url != BASE_URL:
        headers["Referer"] = BASE_URL
        headers["Sec-Fetch-Site"] = "same-origin"
    last_err = None
    attempt = 0
    total_ceiling = RETRIES * len(IMPERSONATE_OPTIONS)
    while attempt < total_ceiling:
        opts = _available_impersonates()
        if not opts:
            print(f"    [fetch fail] {url} -> no supported impersonate profiles in this curl_cffi install")
            return None
        imp = opts[attempt % len(opts)]
        try:
            r = _session_for(imp).get(url, headers=headers)
            if r.status_code == 200 and len(r.content) > 200:
                _polite()
                return r.text
            if r.status_code in (404, 410):
                return None
            last_err = f"[{imp}] HTTP {r.status_code} len={len(r.content)}"
            time.sleep(min(2 ** (attempt // max(len(opts), 1)), 30) + random.random())
        except Exception as e:
            if _is_unsupported_err(e):
                with _BAD_LOCK:
                    if imp not in _BAD_IMPERSONATES:
                        _BAD_IMPERSONATES.add(imp)
                        print(f"    [impersonate] dropping {imp!r}: {e}")
                # retry immediately with a different profile, don't burn a backoff
                attempt += 1
                continue
            last_err = f"[{imp}] {type(e).__name__}: {e}"
            time.sleep(min(2 ** (attempt // max(len(opts), 1)), 30) + random.random())
        attempt += 1
    print(f"    [fetch fail] {url} -> {last_err}")
    return None


## URL Parsing

Regexes and helpers that recognise *subforum* vs. *thread* URLs and pull out
their IDs, slugs and page numbers — the basis for navigating the forum.

In [ ]:
SUBFORUM_INDEX_RE = re.compile(r"^/([a-z0-9-]+?-\d+)/?$", re.I)
SUBFORUM_PAGE_RE  = re.compile(r"^/([a-z0-9-]+?-\d+)-(\d+)\.html$", re.I)
THREAD_RE         = re.compile(r"^/([a-z0-9-]+?)/(\d+)-([^/]+?)(?:-(\d+))?\.html$", re.I)

def is_internal(url: str) -> bool:
    return urlparse(url).netloc.lower() == DOMAIN

def parse_thread_url(url: str):
    m = THREAD_RE.match(urlparse(url).path)
    if not m:
        return None
    sub, tid, slug, page = m.groups()
    return sub, tid, slug, int(page) if page else 1

def parse_subforum_url(url: str) -> Optional[tuple[str, int]]:
    p = urlparse(url).path
    m = SUBFORUM_INDEX_RE.match(p)
    if m:
        return m.group(1), 1
    m = SUBFORUM_PAGE_RE.match(p)
    if m:
        return m.group(1), int(m.group(2))
    return None

## Relevance Filters & HTML Parsers

Keyword matching for the relevance and strict **policy** gates, plus the
BeautifulSoup parsers that pull thread links from listing pages and individual
posts (author, date, body) from thread pages.

In [ ]:
_WORDSPLIT_RE = re.compile(r"[^a-z0-9]+")

def _tokens(s: str) -> list[str]:
    return [t for t in _WORDSPLIT_RE.split(s.lower()) if t]

def _matches_keyword(kw: str, text_lower: str, token_set: set[str]) -> bool:
    """Single-token keyword matches only as a whole token; multi-token keyword
    matches as a contiguous substring (spaces and hyphens both count as breaks)."""
    kw_toks = _tokens(kw)
    if not kw_toks:
        return False
    if len(kw_toks) == 1:
        return kw_toks[0] in token_set

    phrase = "-".join(kw_toks)
    normalized = "-".join(_tokens(text_lower))
    return phrase in normalized

def is_relevant_slug(slug: str) -> bool:
    s = slug.lower()
    toks = set(_tokens(s))
    if any(_matches_keyword(bad, s, toks) for bad in _EXCLUDE_SET):
        return False
    return any(_matches_keyword(kw, s, toks) for kw in _INCLUDE_SET)

def is_relevant_text(text: str) -> bool:
    if not text:
        return False
    s = text.lower()
    toks = set(_tokens(s))
    if any(_matches_keyword(bad, s, toks) for bad in _EXCLUDE_SET):
        return False
    return any(_matches_keyword(kw, s, toks) for kw in _INCLUDE_SET)

def policy_hits(text: str) -> list[str]:
    """Return the list of POLICY_KEYWORDS that match in `text` (slug+title+bodies).
    Excludes the text entirely if any EXCLUDE_KEYWORDS hit."""
    if not text:
        return []
    s = text.lower()
    toks = set(_tokens(s))
    if any(_matches_keyword(bad, s, toks) for bad in _EXCLUDE_SET):
        return []
    return [kw for kw in _POLICY_SET if _matches_keyword(kw, s, toks)]

def is_policy_relevant(text: str) -> bool:
    """Strict Gulf-Council / FEI policy gate. ≥1 POLICY_KEYWORDS match required."""
    return bool(policy_hits(text))
def _matches_allowlist(sub_key: str) -> bool:
    if not SUBFORUM_ALLOWLIST:
        return True
    stem = sub_key.rsplit("-", 1)[0]
    return any(a == sub_key or a == stem or sub_key.startswith(a + "-") for a in SUBFORUM_ALLOWLIST)

def discover_subforums(home_html: str) -> set[str]:
    soup = BeautifulSoup(home_html, "html.parser")
    keys: set[str] = set()
    for a in soup.find_all("a", href=True):
        href = urljoin(BASE_URL, a["href"])
        if not is_internal(href):
            continue
        info = parse_subforum_url(href)
        if info and info[1] == 1 and _matches_allowlist(info[0]):
            keys.add(info[0])
    return keys

def parse_subforum_page(html: str, base_url: str) -> tuple[list[tuple[str, str]], set[str], Optional[str]]:
    """Return (threads [(url, slug)], child_subforum_keys, next_page_url)."""
    soup = BeautifulSoup(html, "html.parser")
    threads: list[tuple[str, str]] = []
    seen_tids: set[str] = set()
    children: set[str] = set()
    for a in soup.find_all("a", href=True):
        href = urljoin(base_url, a["href"])
        href, _ = urldefrag(href)
        if not is_internal(href):
            continue
        info = parse_subforum_url(href)
        if info and info[1] == 1:
            children.add(info[0])
            continue
        tinfo = parse_thread_url(href)
        if tinfo:
            sub, tid, slug, _ = tinfo
            if tid in seen_tids:
                continue
            seen_tids.add(tid)
            canonical = f"https://{DOMAIN}/{sub}/{tid}-{slug}.html"
            threads.append((canonical, slug))
    cur = parse_subforum_url(base_url)
    next_url = None
    if cur:
        sub_key, cur_page = cur
        want = f"/{sub_key}-{cur_page + 1}.html"
        for a in soup.find_all("a", href=True):
            h = urljoin(base_url, a["href"])
            if urlparse(h).path == want:
                next_url = h
                break
    return threads, children, next_url

def _text(el) -> str:
    return el.get_text(" ", strip=True) if el else ""

_DATE_RE = re.compile(r"\d{2}-\d{2}-\d{4}\s*\|?\s*\d{1,2}:\d{2}\s*[AP]M", re.I)

def parse_thread_page(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    title = _text(soup.find("h1"))
    if not title and soup.title:
        t = _text(soup.title)
        title = t.split(" - The Hull Truth")[0].strip() or t

    posts = []
    for container in soup.find_all("div", id=re.compile(r"^post\d+$")):
        m = re.match(r"^post(\d+)$", container.get("id", ""))
        if not m:
            continue
        pid = m.group(1)
        # Author
        a_author = container.find(class_=re.compile(r"\bbigusername\b"))
        author = _text(a_author)
        if not author:
            for sc in container.find_all("script", attrs={"type": "application/ld+json"}):
                try:
                    d = json.loads(sc.string or "{}")
                    if isinstance(d, dict) and d.get("@type") == "Person" and d.get("name"):
                        author = d["name"]
                        break
                except Exception:
                    pass
        # Date
        date_text = ""
        thead = container.find(class_=re.compile(r"\bthead\b"))
        if thead:
            dt_cell = thead.find("div", class_=re.compile(r"\btcell\b"))
            if dt_cell:
                raw = dt_cell.get_text(" ", strip=True)
                mm = _DATE_RE.search(raw)
                date_text = mm.group(0) if mm else raw
        # Body
        msg = container.find(id=f"post_message_{pid}")
        body = _text(msg)
        posts.append({"post_id": pid, "author": author, "date": date_text, "body": body})

    tinfo = parse_thread_url(url)
    next_url = None
    if tinfo:
        _, tid, _, cur_page = tinfo
        want = cur_page + 1
        for a in soup.find_all("a", href=True):
            h = urljoin(url, a["href"])
            ti = parse_thread_url(h)
            if ti and ti[1] == tid and ti[3] == want:
                next_url = h
                break

    return {"title": title, "posts": posts, "next_url": next_url}


## Run State & Text Rendering

Loads and saves resumable crawl progress (`source_registry.json`) and renders a
downloaded thread into a clean, human-readable `.txt` file.

In [ ]:
def load_state() -> dict:

    defaults = {
        "subforums_discovered": [],
        "listings_done":        [],
        "thread_urls":          {},   # sub_key -> [ {url, slug} ]
        "threads_done":         [],
        "threads_skipped":      [],   # irrelevant after body check
    }
    if STATE_FILE.exists():
        try:
            loaded = json.loads(STATE_FILE.read_text(encoding="utf-8"))
            if isinstance(loaded, dict):
                defaults.update(loaded)
        except Exception:
            pass
    return defaults

def save_state(s: dict):
    STATE_FILE.parent.mkdir(parents=True, exist_ok=True)
    STATE_FILE.write_text(json.dumps(s, indent=1, ensure_ascii=False), encoding="utf-8")


# ───────────────────────────── TXT OUTPUT ─────────────────────────────
_SEPARATOR = "=" * 80
_POST_RULE = "-" * 80

def render_thread_txt(thread_data: dict) -> str:
    lines: list[str] = []
    lines.append(f"Title      : {thread_data.get('title','')}")
    lines.append(f"Thread ID  : {thread_data.get('thread_id','')}")
    lines.append(f"Subforum   : {thread_data.get('subforum','')}")
    lines.append(f"URL        : {thread_data.get('url','')}")
    lines.append(f"Posts      : {thread_data.get('post_count',0)}")
    lines.append(f"Pages      : {thread_data.get('pages',0)}")
    lines.append(f"Scraped At : {thread_data.get('scraped_at','')}")
    lines.append(_SEPARATOR)
    lines.append("")
    for idx, post in enumerate(thread_data.get("posts", []), start=1):
        lines.append(f"Post #{idx}")
        lines.append(f"Author : {post.get('author','')}")
        lines.append(f"Date   : {post.get('date','')}")
        lines.append(f"PostID : {post.get('post_id','')}")
        lines.append(_POST_RULE)
        body = (post.get("body") or "").strip()
        lines.append(body if body else "[empty body]")
        lines.append("")
        lines.append(_SEPARATOR)
        lines.append("")
    return "\n".join(lines)


## Crawl Drivers

Walks every listing page of a subforum to collect thread URLs, downloads all
pages of a single thread, and resolves the output file paths for each thread.

In [ ]:
def crawl_subforum_listing(sub_key: str) -> tuple[list[dict], set[str]]:
    """Walk a subforum's listing pages. Returns (thread_entries, child_subforum_keys)
    where each entry is {"url": ..., "slug": ..., "relevant": bool}."""
    url = f"https://{DOMAIN}/{sub_key}/"
    entries: list[dict] = []
    seen: set[str] = set()
    children: set[str] = set()
    pages = 0
    while url and pages < MAX_LIST_PAGES:
        html = fetch(url)
        if not html:
            break
        page_threads, kids, nxt = parse_subforum_page(html, url)
        for c in kids:
            if c != sub_key and _matches_allowlist(c):
                children.add(c)
        for tu, slug in page_threads:
            info = parse_thread_url(tu)
            if not info:
                continue
            tid = info[1]
            if tid in seen:
                continue
            seen.add(tid)
            entries.append({
                "url":      tu,
                "slug":     slug,
                "relevant": is_relevant_slug(slug),
            })
        if nxt == url:
            break
        url = nxt
        pages += 1
    return entries, children

def crawl_thread(thread_url: str) -> Optional[dict]:
    info = parse_thread_url(thread_url)
    if not info:
        return None
    sub, tid, slug, _ = info
    posts = []
    title = ""
    pages = 0
    url = thread_url
    visited: set[str] = set()
    while url and pages < MAX_THREAD_PAGES:
        url, _ = urldefrag(url)
        if url in visited:
            break
        visited.add(url)
        html = fetch(url)
        if not html:
            break
        parsed = parse_thread_page(html, url)
        if not title:
            title = parsed["title"]
        posts.extend(parsed["posts"])
        url = parsed["next_url"]
        pages += 1
    return {
        "thread_id":  tid,
        "subforum":   sub,
        "slug":       slug,
        "url":        thread_url,
        "title":      title,
        "pages":      pages,
        "post_count": len(posts),
        "scraped_at": datetime.now(timezone.utc).isoformat(),
        "posts":      posts,
    }


def txt_path(tid: str) -> Path:
    return RAW_TXT_DIR / f"{tid}.txt"

def json_path(tid: str) -> Path:
    return METADATA_DIR / f"{tid}.json"


## Metadata & Export

Derives `Location` from the subforum, parses the first post's date, applies the
**last-10-years** window, builds the per-thread metadata record, and exports the
consolidated `blogs_forums.csv`.

In [ ]:
_PUBDATE_RE = re.compile(
    r"(\d{2})-(\d{2})-(\d{4})\s*\|?\s*(\d{1,2}):(\d{2})\s*([AP]M)", re.I
)

def location_from_subforum(sub: str) -> str:
    """`gulf-coast-31` -> `Gulf Coast`. Strip trailing numeric id, title-case."""
    if not sub:
        return ""
    parts = sub.rsplit("-", 1)
    stem = parts[0] if len(parts) == 2 and parts[1].isdigit() else sub
    return " ".join(w.capitalize() for w in stem.split("-") if w)

def parse_published_date(thread_data: dict) -> str:
    """First post's date as ISO 8601; fall back to raw string if unparseable."""
    posts = thread_data.get("posts") or []
    if not posts:
        return ""
    raw = (posts[0].get("date") or "").strip()
    if not raw:
        return ""
    m = _PUBDATE_RE.search(raw)
    if not m:
        return raw
    mm, dd, yy, hh, mi, ampm = m.groups()
    hh = int(hh)
    if ampm.upper() == "PM" and hh != 12:
        hh += 12
    elif ampm.upper() == "AM" and hh == 12:
        hh = 0
    try:
        return datetime(int(yy), int(mm), int(dd), hh, int(mi)).isoformat()
    except Exception:
        return raw

def within_data_window(date_str: str, years: int = DATA_WINDOW_YEARS) -> bool:
    """True if `date_str` falls within the last `years` years.
    Pulls the first 4-digit year out of the string (works for ISO 8601 and the
    site's MM-DD-YYYY format) and compares it to the cutoff year. Empty or
    unparseable dates return True so we don't drop content we can't date."""
    if not date_str:
        return True
    m = re.search(r"(?:19|20)\d{2}", date_str)
    if not m:
        return True
    return int(m.group(0)) >= datetime.now().year - years

def build_metadata(thread_data: dict) -> dict:
    """Project a crawled thread into the public metadata schema."""
    return {
        "Document_ID":    thread_data.get("thread_id", ""),
        "Source_Type":    SOURCE_TYPE,
        "Source_Name":    SOURCE_NAME,
        "URL":            thread_data.get("url", ""),
        "Title":          thread_data.get("title", ""),
        "Published_Date": parse_published_date(thread_data),
        "Location":       location_from_subforum(thread_data.get("subforum", "")),
    }
def export_metadata_csv() -> int:
    """Consolidate every per-thread metadata JSON into records/blogs_forums.csv."""
    csv_path = RECORDS_DIR / "blogs_forums.csv"
    rows: list[dict] = []
    for jf in sorted(METADATA_DIR.glob("*.json")):
        try:
            d = json.loads(jf.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(d, dict) or "Document_ID" not in d:
            continue
        rows.append({k: d.get(k, "") for k in METADATA_FIELDS})
    if rows:
        with csv_path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(METADATA_FIELDS), quoting=csv.QUOTE_ALL)
            w.writeheader()
            w.writerows(rows)
    return len(rows)


## Run — Forum Crawl

Ties Part 1 together: discover subforums → collect relevant thread URLs →
download threads (gated by the policy filter **and** the 10-year window) → write
text + metadata → export the CSV and a run summary.

In [ ]:
def main(target: Optional[int] = None):
    global TARGET_THREADS
    if target is None:
        ap = argparse.ArgumentParser(description="The Hull Truth — Gulf Council / fisheries thread scraper.")
        ap.add_argument("-n", "--target", type=int, default=TARGET_THREADS,
                        help=f"target number of relevant threads to download (default {TARGET_THREADS})")
        args, _ = ap.parse_known_args()
        target = args.target
    TARGET_THREADS = max(int(target or 0), 0)
    print(f"[cfg] TARGET_THREADS = {TARGET_THREADS or 'unlimited'}")
    print(f"[cfg] SUBFORUM_ALLOWLIST = {SUBFORUM_ALLOWLIST}")
    print(f"[cfg] RELEVANCE_KEYWORDS = {len(RELEVANCE_KEYWORDS)} terms")

    for p in (DATA, RAW_TXT_DIR, METADATA_DIR, RECORDS_DIR):
        p.mkdir(parents=True, exist_ok=True)
    state = load_state()

    import curl_cffi as _cc
    print(f"[0/3] curl_cffi version: {_cc.__version__}")

    # ---------- Phase 1: discover subforums + collect relevant thread URLs ----------
    print("\n[1/3] Discovering subforums + collecting thread URLs (relevance-filtered)...")
    home = fetch(BASE_URL)
    seeds: list[str] = []
    if home:
        seeds = sorted(discover_subforums(home))
    else:
        print("    [warn] homepage fetch failed — falling back to known gulf-coast seed")
    # Canonical fallback if the homepage didn't expose gulf-coast.
    if not seeds:
        seeds = ["gulf-coast-31"]
    queue: list[str] = list(seeds)
    seen_subs: set[str] = set(queue)
    relevant_urls: list[str] = []
    visited = 0

    while queue:
        sk = queue.pop(0)
        visited += 1
        if sk in state["listings_done"] and sk in state["thread_urls"]:
            entries = state["thread_urls"][sk]
            # Still probe for new child subforums cheaply
            h = fetch(f"https://{DOMAIN}/{sk}/")
            children: set[str] = set()
            if h:
                _, kids, _ = parse_subforum_page(h, f"https://{DOMAIN}/{sk}/")
                children = {c for c in kids if c != sk and _matches_allowlist(c)}
        else:
            entries, children = crawl_subforum_listing(sk)
            state["thread_urls"][sk] = entries
            state["listings_done"] = sorted(set(state["listings_done"] + [sk]))
            save_state(state)

        relevant_here = [e["url"] for e in entries if e.get("relevant")]
        print(f"    [{visited}] {sk}: {len(entries)} threads, {len(relevant_here)} relevant, {len(children)} child subs")
        relevant_urls.extend(relevant_here)
        for c in children:
            if c not in seen_subs:
                seen_subs.add(c)
                queue.append(c)
        if TARGET_THREADS and len(set(relevant_urls)) >= TARGET_THREADS * 2:
            # Gather ~2× the target so we have headroom in case of fetch failures
            print(f"    collected enough candidates ({len(set(relevant_urls))}); stopping discovery")
            break

    state["subforums_discovered"] = sorted(seen_subs)
    save_state(state)

    # De-dup by thread id, preserve order
    seen_tid: set[str] = set()
    uniq: list[str] = []
    for u in relevant_urls:
        info = parse_thread_url(u)
        if not info or info[1] in seen_tid:
            continue
        seen_tid.add(info[1])
        uniq.append(u)
    relevant_urls = uniq
    print(f"    -> {len(seen_subs)} subforums seen, {visited} walked, {len(relevant_urls)} unique relevant URLs")

    # ---------- Phase 2: download threads until we hit TARGET_THREADS ----------
    print("\n[2/3] Downloading threads (relevance-filtered, capped at TARGET_THREADS)...")
    done = set(state["threads_done"])
    skipped = set(state.get("threads_skipped", []))
    # Already-downloaded count toward the target
    already = sum(1 for u in relevant_urls if u in done)
    pending = [u for u in relevant_urls if u not in done and u not in skipped]
    print(f"    already downloaded: {already}   pending candidates: {len(pending)}   target: {TARGET_THREADS}")

    t0 = time.time()
    completed = already
    save_lock = threading.Lock()

    def _submit_and_drain():
        nonlocal completed
        if TARGET_THREADS and completed >= TARGET_THREADS:
            return
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            futures = {pool.submit(crawl_thread, u): u for u in pending}
            for fut in as_completed(futures):
                u = futures[fut]
                if TARGET_THREADS and completed >= TARGET_THREADS:
                    # We've hit the target; drain remaining quickly without recording.
                    for f in futures:
                        f.cancel()
                    break
                try:
                    d = fut.result()
                except Exception as e:
                    print(f"    ! {u} -> {e}")
                    continue
                if not d:
                    continue

                if (d.get("post_count") or 0) == 0:
                    with save_lock:
                        skipped.add(u)
                        state["threads_skipped"] = sorted(skipped)
                    continue
                posts_list = d.get("posts") or []
                op_body = (posts_list[0].get("body") or "") if posts_list else ""
                topic_blob = " ".join([
                    d.get("title") or "",
                    d.get("slug") or "",
                    op_body,
                ])
                hits = policy_hits(topic_blob)
                if not hits:
                    with save_lock:
                        skipped.add(u)
                        state["threads_skipped"] = sorted(skipped)
                    continue

                # Last-N-years window: drop threads whose first post predates it.
                if not within_data_window(parse_published_date(d)):
                    with save_lock:
                        skipped.add(u)
                        state["threads_skipped"] = sorted(skipped)
                    continue

                tid = d["thread_id"]
                # Write TXT (readable: full title + posts)
                tp = txt_path(tid)
                tp.write_text(render_thread_txt(d), encoding="utf-8")
                # Write JSON (metadata only — schema in METADATA_FIELDS)
                meta = build_metadata(d)
                jp = json_path(tid)
                jp.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

                with save_lock:
                    done.add(u)
                    completed += 1
                    if completed % 25 == 0:
                        state["threads_done"] = sorted(done)
                        save_state(state)
                        rate = (completed - already) / max(time.time() - t0, 1e-9)
                        print(f"    {completed}/{TARGET_THREADS}  {rate:.2f} thr/s  last: {(d.get('title') or '')[:60]}")

                if TARGET_THREADS and completed >= TARGET_THREADS:
                    print(f"    reached TARGET_THREADS={TARGET_THREADS}, stopping downloads")
                    for f in futures:
                        f.cancel()
                    break

    _submit_and_drain()

    state["threads_done"]    = sorted(done)
    state["threads_skipped"] = sorted(skipped)
    save_state(state)

    # ---------- Phase 3: export metadata CSV + summary ----------
    print("\n[3/3] Exporting metadata CSV...")
    nt = export_metadata_csv()
    summary = {
        "target_threads":     TARGET_THREADS,
        "threads_written":    nt,
        "subforums_walked":   visited,
        "subforums_seen":     len(seen_subs),
        "candidates_listed":  len(relevant_urls),
        "skipped_irrelevant": len(skipped),
        "finished_at":        datetime.now(timezone.utc).isoformat(),
        "raw_txt_dir":        str(RAW_TXT_DIR.relative_to(ROOT)),
        "metadata_dir":       str(METADATA_DIR.relative_to(ROOT)),
        "records_dir":        str(RECORDS_DIR.relative_to(ROOT)),
    }
    (RECORDS_DIR / "summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    print(f"\nDone. Threads: {nt}")
    print(f"    TXT:      {RAW_TXT_DIR}")
    print(f"    Metadata: {METADATA_DIR}")
    print(f"    CSV:      {RECORDS_DIR / 'blogs_forums.csv'}")
    print(f"    Summary:  {RECORDS_DIR / 'summary.json'}")


if __name__ == "__main__":
    main()


# Part 2 — Multi-Source Blog & Forum Scraper

Breadth-first crawls a curated list of fisheries-policy websites — management
bodies, state wildlife agencies, conservation NGOs, Sea Grant programs, angler
magazines and seafood trade press — and saves only documents that are about
**Gulf policy** *and* published in the **last 10 years**, into the same
`Data_Repository` as Part 1.

## Imports

Standard-library and scraping imports. Playwright and playwright-stealth are
**optional** — imported only if installed, and used solely for JavaScript-
rendered or Cloudflare-protected sources.

In [ ]:
import argparse
import csv
import hashlib
import json
import random
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.parse import urljoin, urlparse, urldefrag

from bs4 import BeautifulSoup
from curl_cffi import requests as cr

try:
    from playwright.sync_api import sync_playwright, TimeoutError as _PWTimeoutError
    _PW_AVAILABLE = True
except ImportError:
    _PW_AVAILABLE = False

try:
    from playwright_stealth import Stealth as _Stealth
    _STEALTH_AVAILABLE = True
except ImportError:
    _STEALTH_AVAILABLE = False

## HTTP Constants & Relevance Keywords

Browser headers, the TLS-fingerprint profile list, and the **strong / weak**
*policy* and *Gulf* keyword sets. A page must hit at least one strong policy
keyword and one strong Gulf keyword to be considered relevant.

In [ ]:
# Browser-like request headers used for all fetches.
BROWSER_HEADERS = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9",
    "Cache-Control": "max-age=0",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Upgrade-Insecure-Requests": "1",
}

IMPERSONATE_OPTIONS = [
    "chrome136", "chrome133a", "chrome131", "chrome124", "chrome123",
    "chrome120", "chrome119", "chrome116", "chrome110", "chrome107",
    "chrome104", "chrome101", "chrome100", "chrome99",
    "edge101", "edge99",
    "safari18_0", "safari17_0", "safari15_5", "safari15_3",
]

def _is_unsupported_err(exc: BaseException) -> bool:
    name = type(exc).__name__.lower()
    msg = str(exc).lower()
    return "impersonate" in name or "not supported" in msg or "unknown impersonate" in msg


# ───────────────────────────── POLICY FILTER ─────────────────────────────
POLICY_KEYWORDS: tuple[str, ...] = (
    # Management bodies / law (precise agency names — no bare "council")
    "gulf-council", "gulf council", "gmfmc", "nmfs", "noaa", "fwc",
    "tpwd", "ldwf", "lwf", "wlf", "sefsc", "magnuson", "msa",
    # Regulatory instruments — high-precision terms only
    "amendment", "regulation", "regulations", "rulemaking",
    "ifq", "quota", "allocation", "fmp", "mrip", "mrfss",
    # Seasons / closures
    "closure", "reopen", "open-season", "closed-season",
    "season-closure", "season-opening",
    # Catch limits (multi-word forms only)
    "bag-limit", "size-limit", "slot-limit", "trip-limit", "vessel-limit",
    # Stock assessment / status
    "stock-assessment", "overfishing", "overfished",
    # Sector classifications
    "for-hire",
    # Bycatch / conservation gear
    "by-catch", "bycatch", "release-mortality",
    "descending-device", "descender", "venting-tool", "barotrauma",
    # Enforcement
    "enforcement", "poaching",
    # Public input
    "public-comment", "comment-period", "public-hearing", "public-meeting",
    "testimony",
    # FEI / Ecosystem-based fishery management
    "ecosystem-based-management", "ebm", "ecosystem-assessment",
    "fishery-management", "fisheries-management",
    "sustainable-fishery", "sustainable-fisheries",
    # Gulf-specific issues with policy weight
    "red-tide",
)

EXCLUDE_KEYWORDS: tuple[str, ...] = (
    "for-sale", "price-reduced", "prop-for-sale", "engine-for-sale",
    "classified", "classifieds",
    "fs", "wtb", "wts", "sold",
)


# filter even though they have nothing to do with the Gulf Council.
GULF_KEYWORDS: tuple[str, ...] = (
    # Geographic — Gulf of Mexico & Gulf states
    "gulf-of-mexico", "gulf of mexico", "gulf of america",
    "gulf-coast", "gulf coast", "gulf-states", "gulf states", "gulf region",
    "northern-gulf", "eastern-gulf", "western-gulf",
    "florida", "alabama", "mississippi", "louisiana", "texas",
    "panhandle", "big-bend", "tampa-bay", "tampa bay", "charlotte-harbor",
    "apalachicola", "pensacola", "mobile-bay", "mobile bay", "biloxi",
    "destin", "venice-louisiana", "port-fourchon", "galveston",
    "port-aransas", "south-padre", "corpus-christi", "freeport",
    "key-west", "florida-keys", "florida keys", "fort-myers",
    "ten-thousand-islands", "everglades",
    # Management bodies / instruments that are Gulf-specific
    "gulf-council", "gulf council", "gmfmc",
    # Gulf-managed species (Reef Fish FMP, CMP FMP, Spiny Lobster, Shrimp,
    # Stone Crab, Coral, Red Drum FMP — all under Gulf Council jurisdiction)
    "red-snapper", "red snapper",
    "gag-grouper", "gag grouper", "red-grouper", "red grouper",
    "black-grouper", "scamp-grouper", "scamp grouper", "yellowedge-grouper",
    "snowy-grouper", "warsaw-grouper", "speckled-hind",
    "vermilion-snapper", "vermilion snapper",
    "gray-triggerfish", "grey-triggerfish", "triggerfish",
    "greater-amberjack", "lesser-amberjack", "amberjack",
    "banded-rudderfish", "almaco-jack",
    "hogfish", "mutton-snapper", "mutton snapper",
    "yellowtail-snapper", "yellowtail snapper",
    "lane-snapper", "queen-snapper", "silk-snapper", "schoolmaster-snapper",
    "blueline-tilefish", "golden-tilefish", "tilefish",
    "king-mackerel", "king mackerel", "kingfish",
    "spanish-mackerel", "spanish mackerel",
    "cobia", "mahi", "dolphinfish",
    "spiny-lobster", "stone-crab", "stone crab",
    "gulf-shrimp", "brown-shrimp", "white-shrimp", "pink-shrimp", "royal-red-shrimp",
    "tarpon", "snook", "redfish", "red-drum", "red drum",
    "speckled-trout", "spotted-seatrout", "spotted seatrout",
    "tripletail", "sheepshead", "pompano",
    # Gulf-specific phenomena
    "red-tide", "deepwater-horizon", "dead-zone", "hypoxia",
    "loop-current",
)


WEAK_POLICY_KEYWORDS: tuple[str, ...] = (
    "regulation", "regulations",
    "closure", "reopen",
    "noaa", "fwc", "tpwd", "ldwf", "lwf", "wlf", "sefsc", "nmfs",
    "enforcement", "poaching",
)


WEAK_GULF_KEYWORDS: tuple[str, ...] = (
    "florida", "fl", "alabama", "al", "mississippi", "ms",
    "louisiana", "la", "texas", "tx",
)

_POLICY_SET  = frozenset(k.lower() for k in POLICY_KEYWORDS)
_EXCLUDE_SET = frozenset(k.lower() for k in EXCLUDE_KEYWORDS)
_GULF_SET    = frozenset(k.lower() for k in GULF_KEYWORDS)
_WEAK_POLICY = frozenset(k.lower() for k in WEAK_POLICY_KEYWORDS)
_WEAK_GULF   = frozenset(k.lower() for k in WEAK_GULF_KEYWORDS)
_STRONG_POLICY_SET = _POLICY_SET - _WEAK_POLICY
_STRONG_GULF_SET   = _GULF_SET   - _WEAK_GULF
_WORDSPLIT_RE = re.compile(r"[^a-z0-9]+")

## Keyword-Matching Helpers

The tokeniser and matchers that report exactly which *policy* and *Gulf*
keywords appear in a page's text — the building blocks of the relevance gates.

In [ ]:

def _tokens(s: str) -> list[str]:
    return [t for t in _WORDSPLIT_RE.split(s.lower()) if t]

def _matches_keyword(kw: str, text_lower: str, token_set: set[str]) -> bool:
    """Single-token keyword matches only as a whole token; multi-token keyword
    matches as a contiguous substring (spaces and hyphens both count as breaks)."""
    kw_toks = _tokens(kw)
    if not kw_toks:
        return False
    if len(kw_toks) == 1:
        return kw_toks[0] in token_set
    phrase = "-".join(kw_toks)
    normalized = "-".join(_tokens(text_lower))
    return phrase in normalized

def policy_hits(text: str) -> list[str]:
    """Return the list of POLICY_KEYWORDS that match in `text`.
    Returns empty if any EXCLUDE_KEYWORDS hit (drop for-sale / classifieds)."""
    if not text:
        return []
    s = text.lower()
    toks = set(_tokens(s))
    if any(_matches_keyword(bad, s, toks) for bad in _EXCLUDE_SET):
        return []
    return [kw for kw in _POLICY_SET if _matches_keyword(kw, s, toks)]

def gulf_hits(text: str) -> list[str]:
    """Return the list of GULF_KEYWORDS that match in `text`. Used as the
    Gulf-relevance gate — a doc must hit ≥1 of these to be saved, otherwise
    we'd accept generic Alaska / Pacific / global fisheries content."""
    if not text:
        return []
    s = text.lower()
    toks = set(_tokens(s))
    return [kw for kw in _GULF_SET if _matches_keyword(kw, s, toks)]



## Location Detection

Infers the dominant Gulf state (Florida, Alabama, Mississippi, Louisiana or
Texas) from a document's text by counting state-specific signal terms, falling
back to `Gulf Council` when none stand out.

In [ ]:
_STATE_SIGNALS: dict[str, tuple[str, ...]] = {
    "Florida": (
        "florida", "fl",
        "pensacola", "tampa", "miami", "jacksonville", "tallahassee",
        "destin", "key-west", "key west", "florida-keys", "florida keys",
        "charlotte-harbor", "charlotte harbor", "apalachicola",
        "panama-city", "panama city", "st-petersburg", "st. petersburg",
        "fort-myers", "fort myers", "naples", "sarasota", "clearwater",
        "everglades", "panhandle", "big-bend", "big bend", "tampa-bay",
        "tampa bay", "ten-thousand-islands", "ten thousand islands",
        "fwc", "myfwc",
    ),
    "Alabama": (
        "alabama", "al",
        "mobile", "mobile-bay", "mobile bay",
        "gulf-shores", "gulf shores", "orange-beach", "orange beach",
        "dauphin-island", "dauphin island", "fort-morgan", "fort morgan",
        "outdoor-alabama",
    ),
    "Mississippi": (
        "mississippi", "ms",
        "biloxi", "gulfport", "pascagoula", "ocean-springs", "ocean springs",
        "bay-st-louis", "bay st louis", "long-beach", "long beach",
        "dmrms", "msdmr",
    ),
    "Louisiana": (
        "louisiana", "la",
        "new-orleans", "new orleans", "venice-louisiana", "venice louisiana",
        "port-fourchon", "port fourchon", "grand-isle", "grand isle",
        "lake-charles", "lake charles", "baton-rouge", "baton rouge",
        "houma", "lafayette", "lwf", "ldwf", "wlf",
    ),
    "Texas": (
        "texas", "tx",
        "galveston", "corpus-christi", "corpus christi",
        "port-aransas", "port aransas", "south-padre", "south padre",
        "freeport", "houston", "brownsville", "matagorda",
        "rockport", "tpwd",
    ),
}

def extract_state(text: str) -> str:
    """Pick the dominant Gulf state mentioned in `text`. Returns one of
    {Florida, Alabama, Mississippi, Louisiana, Texas} when a state can be
    confidently identified, else 'Gulf Council'.

    Strategy: count signal-token hits per state. The state with the most
    hits wins. Ties → the state whose first match appears earliest in the
    lowercased text. Zero total hits → 'Gulf Council'.
    """
    if not text:
        return "Gulf Council"
    s = text.lower()
    toks = set(_tokens(s))
    counts: dict[str, int] = {}
    first_pos: dict[str, int] = {}
    for state, signals in _STATE_SIGNALS.items():
        c = 0
        fp = -1
        for sig in signals:
            if _matches_keyword(sig, s, toks):
                c += 1
                # Track earliest occurrence for tie-break.
                idx = s.find(sig.replace("-", " "))
                if idx < 0:
                    idx = s.find(sig)
                if idx >= 0 and (fp < 0 or idx < fp):
                    fp = idx
        if c:
            counts[state] = c
            first_pos[state] = fp
    if not counts:
        return "Gulf Council"
    max_c = max(counts.values())
    leaders = [st for st, c in counts.items() if c == max_c]
    if len(leaders) == 1:
        return leaders[0]
    # Tie → earliest-appearing wins.
    return min(leaders, key=lambda st: first_pos[st])


## Configuration — Sources, Paths & Targets

The shared `Data_Repository` paths, the master `SOURCES` list of sites to crawl
(grouped by category), the global download target, the per-source page-budget
knobs that control how deep each crawl goes, and the concurrency settings plus
the Cloudflare-aware HTTP `fetch` used by the Part 2 crawl.

In [ ]:
ROOT         = Path.cwd().resolve()
DATA         = ROOT / "Data_Repository"
RAW_TXT_DIR  = DATA / "scraped_text" / "blogs_forums"
METADATA_DIR = DATA / "metadata" / "blogs_forums"
RECORDS_DIR  = DATA / "records"
STATE_FILE   = DATA / "source_registry.json"

METADATA_FIELDS = (
    "Document_ID", "Source_Type", "Source_Name", "URL", "Title",
    "Published_Date", "Location",
)

DATA_WINDOW_YEARS = 10

def within_data_window(date_str: str, years: int = DATA_WINDOW_YEARS) -> bool:
    """True if `date_str` falls within the last `years` years. Pulls the first
    4-digit year out of the string (works for ISO 8601 and most blog date
    formats) and compares it to the cutoff year. Empty or unparseable dates
    return True so we don't drop content we can't date."""
    if not date_str:
        return True
    m = re.search(r"(?:19|20)\d{2}", date_str)
    if not m:
        return True
    return int(m.group(0)) >= datetime.now().year - years


# ───────────────────────────── CONFIG ─────────────────────────────

SOURCES = [

    # ── ADVOCACY / CONSERVATION BLOGS (Gulf-Council & FEI focused) ──────

    {"slug": "cca",  "url": "https://ccaflorida.org/",                    "type": "blog",  "name": "Coastal Conservation Association FL", "location": "Florida", "max_pages": 120},
    {"slug": "ccala","url": "https://ccalouisiana.com/",                  "type": "blog",  "name": "CCA Louisiana",            "location": "Louisiana", "max_pages": 80},
    {"slug": "oc",   "url": "https://oceanconservancy.org/blog/",         "type": "blog",  "name": "Ocean Conservancy",        "location": "Gulf Council", "max_pages": 200},
    {"slug": "sra",  "url": "https://shareholdersalliance.org/",          "type": "blog",  "name": "Gulf Reef Fish Shareholders Alliance", "location": "Gulf Council", "max_pages": 60},
    {"slug": "gwild","url": "https://gulfwild.com/",                      "type": "blog",  "name": "Gulf Wild",                "location": "Gulf Council", "max_pages": 40},
    {"slug": "edf",  "url": "https://www.edf.org/oceans",                 "type": "blog",  "name": "Environmental Defense Fund — Oceans", "location": "Gulf Council", "max_pages": 80},
    {"slug": "pew",  "url": "https://www.pewtrusts.org/en/projects/conserving-marine-life-in-the-united-states", "type": "blog", "name": "Pew — Conserving Marine Life", "location": "Gulf Council", "max_pages": 80},
    {"slug": "hgulf","url": "https://healthygulf.org/",                   "type": "blog",  "name": "Healthy Gulf",             "location": "Gulf Council", "max_pages": 60},
    {"slug": "tnctx","url": "https://www.nature.org/en-us/about-us/where-we-work/united-states/texas/stories-in-texas/", "type": "blog",  "name": "The Nature Conservancy — Texas",     "location": "Texas",       "max_pages": 60},
    {"slug": "tncla","url": "https://www.nature.org/en-us/about-us/where-we-work/united-states/louisiana/", "type": "blog",  "name": "The Nature Conservancy — Louisiana", "location": "Louisiana",   "max_pages": 60},
    {"slug": "tncal","url": "https://www.nature.org/en-us/about-us/where-we-work/united-states/alabama/",   "type": "blog",  "name": "The Nature Conservancy — Alabama",   "location": "Alabama",     "max_pages": 60},
    {"slug": "tncms","url": "https://www.nature.org/en-us/about-us/where-we-work/united-states/mississippi/", "type": "blog",  "name": "The Nature Conservancy — Mississippi", "location": "Mississippi", "max_pages": 60},

    # ── SEA GRANT / RESEARCH INSTITUTES ─────────────────────────────────
    {"slug": "flsg", "url": "https://www.flseagrant.org/",                "type": "blog",  "name": "Florida Sea Grant",        "location": "Florida",  "max_pages": 80},
    {"slug": "gsg",  "url": "https://gulfseagrant.org/",                  "type": "blog",  "name": "Gulf Sea Grant",           "location": "Gulf Council", "max_pages": 60},
    {"slug": "txsg", "url": "https://texasseagrant.org/",                 "type": "blog",  "name": "Texas Sea Grant",          "location": "Texas",    "max_pages": 80},
    {"slug": "hri",  "url": "https://harteresearchinstitute.org/",        "type": "blog",  "name": "Harte Research Institute", "location": "Texas",    "max_pages": 60},
    {"slug": "disl", "url": "https://www.disl.edu/",                      "type": "blog",  "name": "Dauphin Island Sea Lab",   "location": "Alabama",  "max_pages": 60},
    {"slug": "hboi", "url": "https://www.fau.edu/hboi/",                  "type": "blog",  "name": "Harbor Branch Oceanographic Institute (FAU)", "location": "Florida", "max_pages": 60},

    # ── ESTUARY / REGIONAL OCEAN PROGRAMS ───────────────────────────────
    {"slug": "floa", "url": "https://floridaocean.org/",                  "type": "blog",  "name": "Florida Ocean Alliance",   "location": "Florida",  "max_pages": 60},
    {"slug": "tbep", "url": "https://tbep.org/",                          "type": "blog",  "name": "Tampa Bay Estuary Program","location": "Florida",  "max_pages": 60},
    {"slug": "sbep", "url": "https://sarasotabay.org/",                   "type": "blog",  "name": "Sarasota Bay Estuary Program","location": "Florida","max_pages": 60},
    {"slug": "rbnr", "url": "https://rookerybay.org/",                    "type": "blog",  "name": "Rookery Bay NERR",         "location": "Florida",  "max_pages": 60},

    # ── ANGLER MAGAZINES / BLOGS (mixed policy density) ─────────────────
    {"slug": "spf",  "url": "https://www.sportfishingmag.com/",           "type": "blog",  "name": "Sport Fishing Magazine",   "location": "Gulf Council", "max_pages": 60},
    {"slug": "sws",  "url": "https://www.saltwatersportsman.com/",        "type": "blog",  "name": "Salt Water Sportsman",     "location": "Gulf Council", "max_pages": 60},
    {"slug": "otw",  "url": "https://onthewater.com/",                    "type": "blog",  "name": "On The Water",             "location": "Gulf Council", "max_pages": 60},

    # ── INDEPENDENT BLOGS (Florida outdoor + fisheries) ─────────────────
    {"slug": "framb","url": "https://www.floridarambler.com/",             "type": "blog",  "name": "Florida Rambler",          "location": "Florida",  "max_pages": 60},

    # ── INDEPENDENT FISHERIES BLOGS / ANGLER RESOURCES ──────────────────
    {"slug": "tfb",  "url": "https://thefisheriesblog.com/",               "type": "blog",  "name": "The Fisheries Blog",       "location": "Gulf Council", "max_pages": 60},
    {"slug": "fgof", "url": "https://www.floridagofishing.com/",           "type": "blog",  "name": "Florida Go Fishing",       "location": "Florida",  "max_pages": 60},
    {"slug": "btt",  "url": "https://www.bonefishtarpontrust.org/",        "type": "blog",  "name": "Bonefish & Tarpon Trust",  "location": "Florida",  "max_pages": 60},
    {"slug": "ocnb", "url": "https://oceana.org/blog/",                    "type": "blog",  "name": "Oceana Blog",              "location": "Gulf Council", "max_pages": 80},
    {"slug": "surf", "url": "https://www.surfrider.org/coastal-blog",      "type": "blog",  "name": "Surfrider Coastal Blog",   "location": "Gulf Council", "max_pages": 60},
    {"slug": "nrdc", "url": "https://www.nrdc.org/issues/oceans",          "type": "blog",  "name": "NRDC Oceans",              "location": "Gulf Council", "max_pages": 80},
    {"slug": "fwld", "url": "https://www.fishingworld.com/",               "type": "blog",  "name": "Fishing World",            "location": "Gulf Council", "max_pages": 60},
    {"slug": "ftm",  "url": "https://www.fishtalkmag.com/",                "type": "blog",  "name": "Fish Talk Magazine",       "location": "Gulf Council", "max_pages": 60},

    # ── SEAFOOD INDUSTRY TRADE PRESS ────────────────────────────────────

    {"slug": "ucn",  "url": "https://www.undercurrentnews.com/",          "type": "blog",  "name": "Undercurrent News",        "location": "Gulf Council", "max_pages": 80},
    {"slug": "sfs",  "url": "https://www.seafoodsource.com/",             "type": "blog",  "name": "SeafoodSource",            "location": "Gulf Council", "max_pages": 60},
    {"slug": "nf",   "url": "https://www.nationalfisherman.com/",         "type": "blog",  "name": "National Fisherman",       "location": "Gulf Council", "max_pages": 60},
    {"slug": "sav",  "url": "https://www.savingseafood.org/",             "type": "blog",  "name": "Saving Seafood",           "location": "Gulf Council", "max_pages": 60},

    # ── TACKLE / GEAR BLOGS (bycatch & conservation gear content) ───────
    {"slug": "jjig", "url": "https://johnnyjigs.com/",                        "type": "blog", "name": "Johnny Jigs",    "location": "Gulf Council", "max_pages": 60},

    # ── LOCAL FISHING REPORTS ───────────────────────────────────────────
    {"slug": "ffrn", "url": "https://www.fishinfranks.com/",                  "type": "blog", "name": "Fishin' Franks", "location": "Port Charlotte, FL", "max_pages": 12},
    {"slug": "ffrn", "url": "https://www.fishinfranks.com/fishin_report.htm", "type": "blog", "name": "Fishin' Franks", "location": "Port Charlotte, FL", "max_pages": 6},
    {"slug": "ffrn", "url": "https://fishinfranks.com/radio_fishin.htm",      "type": "blog", "name": "Fishin' Franks", "location": "Port Charlotte, FL", "max_pages": 6},

    # ── SECONDARY: state wildlife / fisheries agencies ──────────────────
    {"slug": "fwc",  "url": "https://myfwc.com/fishing/saltwater/",       "type": "blog", "name": "Florida FWC Saltwater",   "location": "Florida",     "max_pages": 80},
    {"slug": "dmrms","url": "https://dmr.ms.gov/",                        "type": "blog", "name": "Mississippi DMR",         "location": "Mississippi", "max_pages": 60},
    {"slug": "tpwd", "url": "https://tpwd.texas.gov/",                    "type": "blog", "name": "Texas Parks & Wildlife",  "location": "Texas",       "max_pages": 100},
    {"slug": "oal",  "url": "https://www.outdooralabama.com/",            "type": "blog", "name": "Outdoor Alabama (ADCNR)", "location": "Alabama",     "max_pages": 80},

    ]



TARGET_THREADS = 8000


PAGE_BUDGET_RATIO = 15


MIN_PAGES_PER_SOURCE = 250


# ───────────────────────────── HTTP ─────────────────────────────
WORKERS_PER_SOURCE = 20

WORKERS_PER_SOURCE_JS = 3

MIN_PAGES_PER_SOURCE_JS = 60
TIMEOUT   = 45
DELAY_MIN = 0.7
DELAY_MAX = 2.0

_thread_local = threading.local()
_BAD_IMPERSONATES: set[str] = set()
_BAD_LOCK = threading.Lock()


def _polite():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def _session(imp: str) -> cr.Session:
    key = f"sess_{imp}"
    s = getattr(_thread_local, key, None)
    if s is None:
        s = cr.Session(impersonate=imp, timeout=TIMEOUT)
        setattr(_thread_local, key, s)
    return s

def fetch(url: str, referer: Optional[str] = None) -> tuple[int, str]:
    """Return (status_code, html). status_code=0 on network error.
    Tries up to 4 impersonate profiles; bails fast on persistent anti-bot."""
    url, _ = urldefrag(url)
    headers = dict(BROWSER_HEADERS)
    if referer:
        headers["Referer"] = referer
        headers["Sec-Fetch-Site"] = "same-origin"
    else:
        headers["Sec-Fetch-Site"] = "none"
    profiles = [o for o in IMPERSONATE_OPTIONS if o not in _BAD_IMPERSONATES][:4]
    last_status = 0
    for imp in profiles:
        try:
            r = _session(imp).get(url, headers=headers, allow_redirects=True)
            last_status = r.status_code
            if r.status_code == 200 and len(r.content) > 500:
                _polite()
                return r.status_code, r.text
            if r.status_code in (403, 406, 409, 503):
                continue
            if r.status_code in (404, 410):
                return r.status_code, ""
        except Exception as e:
            if _is_unsupported_err(e):
                with _BAD_LOCK:
                    _BAD_IMPERSONATES.add(imp)
                continue
            time.sleep(0.4)
    return last_status, ""


## Fetcher (Playwright)

A headless-Chromium fetcher with stealth patches for JavaScript-rendered or
Cloudflare-protected sources, plus a router that picks the plain-HTTP or the JS
fetcher per source.

In [ ]:
def _js_browser():
    """Return a (Chromium browser, Stealth) pair owned by the current thread."""
    if not _PW_AVAILABLE:
        raise RuntimeError("playwright is not installed (pip install playwright; "
                           "python -m playwright install chromium)")
    b = getattr(_thread_local, "js_browser", None)
    if b is None:
        pw = sync_playwright().start()
        b = pw.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"],
        )
        # Keep the playwright handle alive too so it doesn't get GC'd.
        setattr(_thread_local, "js_playwright", pw)
        setattr(_thread_local, "js_browser", b)
        # Stealth instance is per-thread but stateless — safe to reuse.
        setattr(_thread_local, "js_stealth",
                _Stealth() if _STEALTH_AVAILABLE else None)
    return b


def js_fetch(url: str, referer: Optional[str] = None) -> tuple[int, str]:
    """Playwright equivalent of fetch() for JS-rendered pages.
    Applies playwright-stealth patches to bypass Cloudflare's headless-browser
    detection (navigator.webdriver, missing chrome.runtime, etc.).
    Returns (status, html). Status=0 on any exception."""
    url, _ = urldefrag(url)
    try:
        browser = _js_browser()
    except Exception:
        return 0, ""
    stealth = getattr(_thread_local, "js_stealth", None)
    ctx = browser.new_context(
        user_agent=("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/131.0.0.0 Safari/537.36"),
        viewport={"width": 1280, "height": 1024},
        extra_http_headers={"Referer": referer} if referer else {},
    )
    if stealth is not None:
        try:
            stealth.apply_stealth_sync(ctx)
        except Exception:
            pass  # stealth is best-effort; continue without it on failure
    try:
        page = ctx.new_page()
        try:
            resp = page.goto(url, wait_until="domcontentloaded", timeout=30_000)
            status = resp.status if resp else 0
            if status >= 400:
                return status, ""
            # Best-effort wait for JS-loaded content. Don't fail if the page
            # keeps a long-poll open past networkidle's window.
            try:
                page.wait_for_load_state("networkidle", timeout=8_000)
            except _PWTimeoutError:
                pass
            html = page.content()
            _polite()
            return status or 200, html
        finally:
            page.close()
    except _PWTimeoutError:
        return 0, ""
    except Exception:
        return 0, ""
    finally:
        ctx.close()


def _route_fetch(src: dict, url: str, ref: Optional[str]) -> tuple[int, str]:
    """Pick the right fetcher based on src['fetcher']."""
    if src.get("fetcher") == "js":
        return js_fetch(url, referer=ref)
    return fetch(url, referer=ref)


## Content Extraction

Extracts the title, published date and clean main text from a page — preferring
`<article>` / `<main>` / forum post bodies, stripping navigation and boilerplate,
and rejecting link-list / index pages so only real articles survive.

In [ ]:
_STRIP_TAGS = ("script", "style", "noscript", "nav", "footer", "header",
               "aside", "form", "iframe", "svg", "button")

# Whole lines to drop verbatim (case-insensitive, exact match).
_BOILERPLATE_TEXT_RE = re.compile(
    r"^(home|sign in|log in|register|create account|menu|search|skip to content|"
    r"toggle navigation|next page|previous page|older posts|newer posts|"
    r"older post|newer post|cookie policy|all rights reserved|"
    r"share this|print this page|email this|tweet|subscribe|"
    r"leave a (?:comment|reply)|reply|comments? \(\d+\)|"
    r"read more|continue reading|view (?:all|more|details)|see (?:all|more)|"
    r"back to top|return to top|scroll down|scroll up)$",
    re.I,
)

# Lines CONTAINING any of these phrases are navigation noise — drop them.
_DROP_LINE_SUBSTR_RE = re.compile(
    r"\b(click here(?:\s+to\s+\w+)?|read more|continue reading|"
    r"view (?:all|more|details)|see (?:all|more)|tap here|"
    r"download (?:here|now|the (?:app|pdf))|listen (?:here|now))\b",
    re.I,
)

# Lines that are *only* a metadata token (episode number, date, hashtag, etc.).
_METADATA_ONLY_LINE_RE = re.compile(
    r"^(?:"
    r"#\d{1,5}"                                  # episode/issue number: #971
    r"|\d{1,2}[/.\-]\d{1,2}[/.\-]\d{2,4}"        # bare date: 05/22/26
    r"|\d{1,2}[/.\-]\d{1,2}"                     # bare partial date: 5/22
    r"|page \d+"                                  # "Page 3"
    r"|\d+ of \d+"                                # "3 of 12"
    r"|\d+ comments?"                             # "5 comments"
    r")$",
    re.I,
)

_EPISODE_PREFIX_RE = re.compile(r"^#\d{1,5}\b")

# A short line containing a date-ish fragment is almost certainly an
# index/archive entry, not real article content.
_DATEY_FRAGMENT_RE = re.compile(r"\d{1,2}[/.\-]\d{1,2}[/.\-]\d{2,4}")

def _strip_html(soup: BeautifulSoup):
    for t in soup.find_all(_STRIP_TAGS):
        t.decompose()

def extract_title(soup: BeautifulSoup) -> str:
    # OG title beats <title> beats h1
    og = soup.find("meta", property="og:title")
    if og and og.get("content"):
        return og["content"].strip()
    if soup.title and soup.title.string:
        t = soup.title.string.strip()
        return re.split(r"\s+[\|·\-–—]\s+", t)[0][:300]
    h1 = soup.find("h1")
    if h1:
        return h1.get_text(strip=True)[:300]
    return ""

def extract_published_date(soup: BeautifulSoup) -> str:
    # OG / article-published / <time datetime=>
    for sel in [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "pubdate"}),
        ("meta", {"name": "publishdate"}),
        ("meta", {"name": "date"}),
        ("meta", {"itemprop": "datePublished"}),
    ]:
        m = soup.find(*sel)
        if m and m.get("content"):
            return m["content"].strip()
    t = soup.find("time")
    if t:
        dt = t.get("datetime") or t.get_text(strip=True)
        if dt:
            return dt.strip()
    return ""

def extract_main_text(soup: BeautifulSoup) -> str:
    """Heuristic main-content extraction. Prefer <article>, then <main>,
    then the densest <div>; fall back to whole body."""
    _strip_html(soup)
    # Best: <article>
    cand = soup.find("article")
    if cand and len(cand.get_text(strip=True)) > 400:
        return _clean_text(cand)
    # Next: <main>
    cand = soup.find("main")
    if cand and len(cand.get_text(strip=True)) > 400:
        return _clean_text(cand)
    # Forum thread bodies — XenForo
    posts = soup.select("div.bbWrapper")
    if posts:
        return "\n\n".join(_clean_text(p) for p in posts if p.get_text(strip=True))
    # Densest div: pick the div with most <p> text by character count
    best = None
    best_len = 0
    for d in soup.find_all("div"):
        txt = d.get_text(" ", strip=True)
        # weight by paragraph count to avoid huge sidebar divs
        p_count = len(d.find_all("p"))
        score = len(txt) + p_count * 200
        if score > best_len and len(txt) > 400:
            best_len, best = score, d
    if best:
        return _clean_text(best)
    return _clean_text(soup.body) if soup.body else ""

def _clean_text(node) -> str:
    """Extract visible text, dropping nav/boilerplate junk.

    Filters applied per line:
      - exact-match boilerplate phrases (sign in, share this, ...)
      - lines CONTAINING nav phrases (click here, read more, ...)
      - metadata-only lines (#971, 05/22/26, "Page 3", ...)
      - lines under 3 chars

    Then a final navigation-density check: if more than half the surviving
    lines are short (<25 chars), the block is really a link list — return
    empty string so the caller's 400-char threshold rejects this page.
    """
    if node is None:
        return ""
    lines: list[str] = []
    for ln in node.get_text("\n", strip=True).split("\n"):
        ln = ln.strip()
        if not ln or len(ln) < 3:
            continue
        if _BOILERPLATE_TEXT_RE.match(ln):
            continue
        if _METADATA_ONLY_LINE_RE.match(ln):
            continue
        if _DROP_LINE_SUBSTR_RE.search(ln):
            continue
        if _EPISODE_PREFIX_RE.match(ln):
            # "#786 his week Fishin- 127 05/07/20" — archive entry
            continue
        if len(ln) < 70 and _DATEY_FRAGMENT_RE.search(ln):
            # short line dominated by a date — archive/index entry
            continue
        lines.append(ln)
    if not lines:
        return ""
    # Navigation-density gate: a real article's lines are mostly substantive
    # paragraphs. A page that's 50%+ short snippets is a link list/index.
    short = sum(1 for ln in lines if len(ln) < 25)
    if len(lines) >= 8 and short / len(lines) > 0.5:
        return ""
    return "\n".join(lines)


## Crawl Engine

Parallel breadth-first crawl within each source. A page is saved only if it
clears **all** gates: a strong policy keyword, a strong Gulf keyword, and the
last-10-years window. A shared global counter stops the run once the overall
target is reached.

In [ ]:
_URL_BAD_EXT = re.compile(r"\.(jpg|jpeg|png|gif|webp|svg|ico|css|js|pdf|zip|mp3|mp4|mov|woff2?|ttf|eot|xml|rss|json)(\?|$)", re.I)
_URL_BAD_PATH = re.compile(r"/(login|register|signup|signin|logout|cart|checkout|account|profile|members?/|tag/|tags/|page/\d+|wp-(?:admin|content|includes|json))(/|$)", re.I)

def discover_links(html: str, base_url: str) -> list[str]:
    """Internal links from the page worth following."""
    soup = BeautifulSoup(html, "html.parser")
    base_host = urlparse(base_url).netloc.lower()
    out: set[str] = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href or href.startswith(("#", "mailto:", "tel:", "javascript:")):
            continue
        absu = urljoin(base_url, href)
        absu, _ = urldefrag(absu)
        u = urlparse(absu)
        if u.scheme not in ("http", "https"):
            continue
        if u.netloc.lower() != base_host:
            continue
        if _URL_BAD_EXT.search(u.path):
            continue
        if _URL_BAD_PATH.search(u.path):
            continue
        out.add(absu)
    return sorted(out)

class _GlobalProgress:
    """Thread-safe counter shared across sources so the crawl can stop as
    soon as TARGET_THREADS docs have been saved across the whole run."""

    def __init__(self, target: int):
        self.target = target
        self.count = 0
        self._lock = threading.Lock()

    def bump(self) -> bool:
        with self._lock:
            self.count += 1
            return self.count >= self.target

    def done(self) -> bool:
        with self._lock:
            return self.count >= self.target

    def remaining(self) -> int:
        with self._lock:
            return max(0, self.target - self.count)


def _process_page(src: dict, url: str, ref: Optional[str]
                  ) -> tuple[int, str, list[str], Optional[dict]]:
    """Worker payload: fetch one URL, extract content, return
    (status, url, child_links, doc_or_None)."""
    status, html = _route_fetch(src, url, ref)
    if status != 200 or not html:
        return status, url, [], None
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return status, url, [], None
    title = extract_title(soup)
    pub = extract_published_date(soup)
    text = extract_main_text(soup)
    doc: Optional[dict] = None
    if title and text and len(text) >= 400:
        blob = f"{title}\n{text}"
        phits = policy_hits(blob)
        ghits = gulf_hits(blob) if phits else []

        strong_p = [h for h in phits if h in _STRONG_POLICY_SET]
        strong_g = [h for h in ghits if h in _STRONG_GULF_SET]
        # Last-N-years window: drop docs whose published date predates it.
        if strong_p and strong_g and within_data_window(pub):
            doc_id = f"{src['slug']}_{hashlib.md5(url.encode('utf-8')).hexdigest()[:10]}"

            doc = {
                "Document_ID":    doc_id,
                "Source_Type":    src["type"],
                "Source_Name":    src["name"],
                "URL":            url,
                "Title":          title[:500],
                "Published_Date": pub,
                "Location":       extract_state(blob),
                "_text":          text,
                "_hits":          phits,
                "_gulf_hits":     ghits,
            }
    return status, url, discover_links(html, url), doc


def crawl_source(src: dict, target_per_source: int,
                 gp: Optional[_GlobalProgress] = None) -> list[dict]:
    """Level-by-level parallel BFS within one source. Fetches up to
    WORKERS_PER_SOURCE pages concurrently per level. Stops when:
      - target_per_source docs have been saved, OR
      - the global progress counter reaches TARGET_THREADS, OR
      - the page budget runs out, OR
      - the site has blocked us 5+ times in a row.
    """
    seed = src["url"]
    is_js = src.get("fetcher") == "js"
    workers = WORKERS_PER_SOURCE_JS if is_js else WORKERS_PER_SOURCE
    pages_floor = MIN_PAGES_PER_SOURCE_JS if is_js else MIN_PAGES_PER_SOURCE

    base_max = src.get("max_pages", 30)
    if target_per_source:
        max_pages = max(base_max, target_per_source * PAGE_BUDGET_RATIO,
                        pages_floor)
    else:
        max_pages = max(base_max, pages_floor)

    visited: set[str] = {seed}
    docs: list[dict] = []
    current_level: list[tuple[str, Optional[str]]] = [(seed, None)]
    blocked_count = 0
    # Per-source diagnostics so silent zeros become legible.
    stats = {"ok": 0, "blocked": 0, "notfound": 0, "err": 0,
             "no_content": 0, "no_hit": 0}

    def at_limit() -> bool:
        if target_per_source and len(docs) >= target_per_source:
            return True
        if gp is not None and gp.done():
            return True
        return False

    with ThreadPoolExecutor(max_workers=workers) as pool:

        while current_level and not at_limit():
            if blocked_count >= 5:
                print(f"    [{src['slug']}] blocked {blocked_count}x — aborting source")
                break

            futures = [pool.submit(_process_page, src, u, r)
                       for u, r in current_level]
            next_level: list[tuple[str, Optional[str]]] = []
            for fut in as_completed(futures):
                try:
                    status, url, children, doc = fut.result()
                except Exception:
                    stats["err"] += 1
                    continue
                if status == 200:
                    stats["ok"] += 1
                    if doc is None:

                        stats["no_hit" if children else "no_content"] += 1
                elif status in (403, 406, 409, 503):
                    blocked_count += 1
                    stats["blocked"] += 1
                elif status in (404, 410):
                    stats["notfound"] += 1
                else:
                    stats["err"] += 1
                if doc is not None:
                    docs.append(doc)
                    if gp is not None:
                        gp.bump()
                if at_limit():
                    # Drain remaining futures but stop enqueueing new work.
                    continue
                for child in children:
                    if child in visited:
                        continue
                    if len(visited) >= max_pages:
                        break
                    visited.add(child)
                    next_level.append((child, url))
            current_level = next_level

    pages = len(visited)
    print(f"    [{src['slug']}] crawled={pages}  "
          f"200={stats['ok']} 403={stats['blocked']} 404={stats['notfound']} err={stats['err']}  "
          f"no-content={stats['no_content']} no-policy={stats['no_hit']} saved={len(docs)}")
    return docs


## Output & State

Writes each accepted document's `.txt` and `.json`, rebuilds the consolidated
`blogs_forums.csv` from every metadata file (Part 1 **and** Part 2), and loads /
saves resumable run state.

In [ ]:
def render_doc_txt(doc: dict) -> str:
    lines = [
        f"Document_ID  : {doc['Document_ID']}",
        f"Source_Type  : {doc['Source_Type']}",
        f"Source_Name  : {doc['Source_Name']}",
        f"URL          : {doc['URL']}",
        f"Title        : {doc['Title']}",
        f"Published    : {doc.get('Published_Date','')}",
        f"Location     : {doc['Location']}",
        "=" * 80,
        "",
        doc.get("_text", ""),
    ]
    return "\n".join(lines)

def write_doc(doc: dict):
    tid = doc["Document_ID"]
    (RAW_TXT_DIR / f"{tid}.txt").write_text(render_doc_txt(doc), encoding="utf-8")
    meta = {k: doc.get(k, "") for k in METADATA_FIELDS}
    (METADATA_DIR / f"{tid}.json").write_text(
        json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
    )

def rebuild_csv() -> int:
    """Read every metadata JSON (from any pipeline) and rebuild blogs_forums.csv."""
    csv_path = RECORDS_DIR / "blogs_forums.csv"
    rows: list[dict] = []
    for jf in sorted(METADATA_DIR.glob("*.json")):
        try:
            d = json.loads(jf.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(d, dict) or "Document_ID" not in d:
            continue
        rows.append({k: d.get(k, "") for k in METADATA_FIELDS})
    if rows:
        with csv_path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(METADATA_FIELDS), quoting=csv.QUOTE_ALL)
            w.writeheader()
            w.writerows(rows)
    return len(rows)


# ───────────────────────────── STATE ─────────────────────────────
# Resumable run state in Data_Repository/source_registry.json (completed sources
# + per-source stats). Loaded defensively — merged onto defaults — because this
# file is shared with the hulltruth pipeline, which stores a different shape.
def load_state() -> dict:

    defaults = {"completed_sources": [], "source_stats": {}}
    if STATE_FILE.exists():
        try:
            loaded = json.loads(STATE_FILE.read_text(encoding="utf-8"))
            if isinstance(loaded, dict):
                defaults.update(loaded)
        except Exception:
            pass
    return defaults

def save_state(s: dict):
    STATE_FILE.parent.mkdir(parents=True, exist_ok=True)
    STATE_FILE.write_text(json.dumps(s, indent=1, ensure_ascii=False), encoding="utf-8")


# ───────────────────────────── RUNNER ─────────────────────────────
# Orchestration: crawl every configured source (in randomized order) until
# TARGET_THREADS docs are saved, write each out, and rebuild the CSV. run() is
# the programmatic entry point; main() wraps it with CLI flags.

def run(target_total: int = TARGET_THREADS,
        only: str = "",
        rebuild_csv_only: bool = False) -> int:
    """Programmatic entry point — call directly from a Colab cell.

    Crawls every configured source (or just the `only=` slugs) until
    `target_total` policy-relevant docs are saved across the whole run.
    Returns the total number of rows in blogs_forums.csv.
    """
    for p in (DATA, RAW_TXT_DIR, METADATA_DIR, RECORDS_DIR):
        p.mkdir(parents=True, exist_ok=True)

    if rebuild_csv_only:
        n = rebuild_csv()
        print(f"Rebuilt blogs_forums.csv with {n} rows from existing JSONs.")
        return n

    state = load_state()
    only_set = {s.strip() for s in only.split(",") if s.strip()}
    sources = [s for s in SOURCES if not only_set or s["slug"] in only_set]
    if not sources:
        print("No sources matched.")
        return 0


    random.shuffle(sources)

    target_per_source = max(1, -(-target_total // len(sources)))
    gp = _GlobalProgress(target_total)

    print(f"[cfg] TARGET_THREADS    = {target_total}")
    print(f"[cfg] sources           = {len(sources)} (order randomized)")
    print(f"[cfg] target / source   = {target_per_source} (auto-distributed)")
    print(f"[cfg] workers / source  = {WORKERS_PER_SOURCE}")
    print(f"[cfg] page budget ratio = {PAGE_BUDGET_RATIO}x per expected doc")
    print()

    summary = []
    for i, src in enumerate(sources, 1):

        per_source_now = target_per_source
        print(f"[{i}/{len(sources)}] {src['slug']:<6} {src['name']} — {src['url']}")
        print(f"           target this source: {per_source_now}   global so far: {gp.count}")
        t0 = time.time()
        try:
            docs = crawl_source(src, per_source_now, gp=None)
        except KeyboardInterrupt:
            raise
        except Exception as e:
            print(f"    EXCEPTION: {type(e).__name__}: {e}")
            docs = []
        for d in docs:
            write_doc(d)
        elapsed = time.time() - t0
        print(f"    -> {len(docs)} policy-relevant docs in {elapsed:.1f}s")
        if docs:
            for d in docs[:5]:
                # Sanitize title for Windows cp1252 console — strip unicode
                # line separators / unprintable chars that crash print().
                title = d['Title'][:70].encode('ascii', 'replace').decode('ascii')
                print(f"       {d['Document_ID']}  {title}")
                print(f"          hits: {d['_hits'][:8]}")
        state["source_stats"][src["slug"] + "@" + src["url"]] = {
            "url": src["url"],
            "docs_saved": len(docs),
            "elapsed_s": round(elapsed, 1),
            "at": datetime.now(timezone.utc).isoformat(),
        }
        save_state(state)
        summary.append((src["slug"], src["url"], len(docs)))
        print()

    total_rows = rebuild_csv()
    saved_this_run = sum(n for _, _, n in summary)
    print(f"=== blogs_forums.csv rebuilt: {total_rows} total rows ===")
    print(f"=== this run: {saved_this_run} docs across {sum(1 for _,_,n in summary if n)} of {len(summary)} sources ===")
    print("Per-source results:")
    for slug, url, n in summary:
        marker = "OK " if n else "-- "
        print(f"  {marker} {n:>3}  {slug:<6}  {url}")
    return total_rows

## Run — Multi-Source Crawl

Entry point for Part 2: crawls every configured source (in randomised order)
until the global target is met, writes the results, and rebuilds the CSV.

In [ ]:
def main():
    ap = argparse.ArgumentParser(description="Multi-source Gulf-Council / FEI scraper")
    ap.add_argument("-n", "--target-threads", type=int, default=TARGET_THREADS,
                    help=f"total policy-relevant docs to save across ALL sources (default {TARGET_THREADS})")
    ap.add_argument("--only", type=str, default="",
                    help="comma-separated source slugs to run (e.g. fpost,ffrn)")
    ap.add_argument("--rebuild-csv-only", action="store_true",
                    help="skip crawling, just rebuild blogs_forums.csv from existing JSONs")
    args, _unknown = ap.parse_known_args()
    run(target_total=args.target_threads,
        only=args.only,
        rebuild_csv_only=args.rebuild_csv_only)


if __name__ == "__main__":
    main()
